# NVIDIA Ising integration

qb-compiler 0.4.0 adds `qb_compiler.ising` — the first Qiskit-side onramp to NVIDIA's `Ising-Decoder-SurfaceCode-1` model family (released 2026-04-14).

This notebook walks through:

1. Declaring a rotated surface-code memory experiment via `SurfaceCodePatchSpec`.
2. Producing the 4-channel `(B, 4, T, D, D)` float32 tensor the pretrained Ising decoder consumes.
3. Running the bundled PyMatching baseline and collecting LER statistics.
4. Plugging in the optional NVIDIA pre-decoder (user supplies gated-HF weights; qb-compiler does not redistribute them).

Install: `pip install qb-compiler[ising]` (adds stim + pymatching).

For the NVIDIA pre-decoder: `pip install qb-compiler[ising-nvidia]` (adds torch + safetensors).

In [ ]:
from qb_compiler.ising import (
    SurfaceCodePatchSpec,
    PyMatchingDecoder,
    evaluate_logical_error_rate,
    sample_and_build_tensor,
)

# A d=5 rotated surface-code memory experiment in the X basis.
spec = SurfaceCodePatchSpec(distance=5, rounds=5, basis="X", p_error=0.003)
print(spec)
print(f"num_data_qubits       = {spec.num_data_qubits}")
print(f"num_ancillas_per_basis = {spec.num_ancillas_per_basis}")
print(f"tensor_shape           = {spec.tensor_shape}  # (channels, rounds, rows, cols)")

## 1. Build the 4-channel tensor

The tensor carries:
- Channel 0 (`x_type`): parity flips of X-type detectors, scattered onto a `d x d` grid.
- Channel 1 (`z_type`): parity flips of Z-type detectors.
- Channel 2 (`x_present`): presence mask (bulk = 1.0, boundary = 0.5, zeroed on wrong-basis first/last rounds).
- Channel 3 (`z_present`): same for Z stabilisers.

In [ ]:
tensor, obs = sample_and_build_tensor(spec, shots=128, seed=42)
print(f"tensor shape: {tensor.shape}  dtype: {tensor.dtype}")
print(f"observables:  {obs.shape}  dtype: {obs.dtype}")

print(f"x_type mean (noisy): {tensor[:, 0].mean():.4f}")
print(f"z_type mean (wrong-basis endpoints zeroed): {tensor[:, 1].mean():.4f}")
print(f"x_presence sum per shot: {tensor[0, 2].sum():.1f}  "
      f"(expected: bulk * 1.0 + boundary * 0.5) * rounds)")
print(f"z_presence sum per shot: {tensor[0, 3].sum():.1f}  "
      f"(first and last rounds zeroed for wrong basis)")

## 2. PyMatching baseline

The MWPM baseline is built directly from the stim detector-error model. This is the number any pre-decoder needs to beat.

In [ ]:
decoder = PyMatchingDecoder(spec)
result = evaluate_logical_error_rate(
    spec,
    decoder,
    shots=20_000,
    seed=2026,
    decoder_name="pymatching_baseline",
)
print(f"d={result.distance} T={result.rounds} basis={result.basis} p={result.p_error}")
print(f"  LER = {result.rate:.4e} +/- {result.standard_error:.2e}")
print(f"  errors: {result.logical_errors} / {result.shots}")

## 3. Distance scaling

Classic error-suppression check: at physical error rates below threshold (~1% for rotated surface codes), deeper codes give exponentially lower logical error rates.

In [ ]:
for d in (3, 5, 7):
    spec_d = SurfaceCodePatchSpec(distance=d, rounds=d, basis="X", p_error=0.003)
    dec_d = PyMatchingDecoder(spec_d)
    r = evaluate_logical_error_rate(spec_d, dec_d, shots=20_000, seed=7)
    print(f"d={d}: LER = {r.rate:.4e} +/- {r.standard_error:.2e}")

## 4. Plug in the NVIDIA Ising pre-decoder (optional)

The NVIDIA `Ising-Decoder-SurfaceCode-1` pre-decoder emits a 4-channel correction tensor; qb-compiler XORs it onto the input syndromes and feeds the residual to PyMatching.

To run this cell you need:

1. `pip install qb-compiler[ising-nvidia]` for torch + safetensors.
2. A gated-HuggingFace weights download from `nvidia/Ising-Decoder-SurfaceCode-1-Fast` (or `-Accurate`) — see https://huggingface.co/nvidia/Ising-Decoder-SurfaceCode-1-Fast
3. A `build_model(spec)` callable sourced from NVIDIA's public Apache-2.0 repo at https://github.com/NVIDIA/Ising-Decoding (the model definition lives at `code/model/predecoder.py`). qb-compiler does NOT vendor it.

The cell below shows the API; it will raise an informative error if `build_model` is not supplied.

In [ ]:
from qb_compiler.ising import IsingDecoderConfig, IsingDecoderWrapper

config = IsingDecoderConfig(
    weights_path="/path/to/Ising-Decoder-SurfaceCode-1-Fast.pt",  # user-supplied
    device="cpu",
    build_model=None,  # replace with NVIDIA's PreDecoderModelMemory_v1 factory
)

try:
    ising = IsingDecoderWrapper(spec, config)
    ising_result = evaluate_logical_error_rate(spec, ising, shots=20_000, seed=2026)
    print(f"NVIDIA Ising + PyMatching residual: LER = {ising_result.rate:.4e}")
except NotImplementedError as exc:
    print("Expected (no build_model supplied):")
    print(f"  {exc}")

## Summary

| Integration level | API | Install |
|-------------------|-----|---------|
| Surface-code spec | `SurfaceCodePatchSpec` | base |
| Tensor builder | `build_ising_tensor` / `sample_and_build_tensor` | `qb-compiler[ising]` |
| PyMatching baseline | `PyMatchingDecoder` | `qb-compiler[ising]` |
| NVIDIA pre-decoder + residual MWPM | `IsingDecoderWrapper` | `qb-compiler[ising-nvidia]` + user-supplied weights |
| LER evaluation harness | `evaluate_logical_error_rate` | `qb-compiler[ising]` |

Stim-simulated results establish the PyMatching baseline. Hardware surface-code validation will follow once dedicated hardware runs are available; existing repetition-code data cannot be consumed by `Ising-Decoder-SurfaceCode-1` weights.

See `src/qb_compiler/ising/README.md` for the licensing breakdown and the orientation-fingerprint caveat when pairing qb-compiler tensors with NVIDIA pretrained weights.